# Project 4 - Neural Radiance Field (NeRF)

In [7]:
# load libraries
import numpy as np
import skimage as sk
import skimage.io as skio
from skimage import img_as_ubyte
import matplotlib.pyplot as plt
from skimage.transform import resize

import cv2
import numpy as np
import glob # for loading multiple files
import viser # visualizations

## Part 0: Calibrating Your Camera and Capturing a 3D Scan

### Part 0.1: Calibrating Your Camera

In [8]:
# calibration pipeline

# starter code from spec
# import cv2
# import numpy as np

# Create ArUco dictionary and detector parameters (4x4 tags)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_params = cv2.aruco.DetectorParameters()

# tag size
tag_size = 0.06 # meters, 0.02 = 2 cm
# 3d coordinates of aruco tag in world frame
obj_pts_single = np.array([
    [0, 0, 0],           # Top-left
    [tag_size, 0, 0],    # Top-right
    [tag_size, tag_size, 0],  # Bottom-right
    [0, tag_size, 0]     # Bottom-left
], dtype = np.float32)

obj_pts = [] # all 3D pts in world frame
img_pts = [] # all 2D pts in image frame

# images = glob.glob('calibration-images/*.JPG') # path to images
images = glob.glob('desk-calibration-images/*.jpg') # path to images, path file ending is case sensitive (e.g. jpg vs JPG are different)

for path in images:
    # load image
    im = cv2.imread(path)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

    # Detect ArUco markers in an image
    # Returns: corners (list of numpy arrays), ids (numpy array)
    corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params) # modify to take in grayscale image

    # Check if any markers were detected
    if ids is None or len(ids) == 0:
        print(f"No ArUco detected in {path}, skipping.")
        continue
    # Process the detected corners
    # corners: list of length N (number of detected tags)
    #   - each element is a numpy array of shape (1, 4, 2) containing the 4 corner coordinates (x, y)
    # ids: numpy array of shape (N, 1) containing the tag IDs for each detected marker
    # Example: if 3 tags detected, corners will be a list of 3 arrays, ids will be shape (3, 1)
        # loop through all detected markers
    for c in corners:
        img_pts.append(c.reshape(4, 2)) # reshape from (1, 4, 2) to shape: (4,2)
        obj_pts.append(obj_pts_single) # append the same 3D points for each detected marker

# 5. calibrate camera
if len(obj_pts) >= 10:
    print(f"Using {len(obj_pts)} detected tags for calibration")
    img_shape = gray.shape[::-1] # (width, height)
    
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        obj_pts,
        img_pts,
        img_shape,
        None, # initial camera matrix (estimated by OpenCV)
        None # initial distortion coefficients
    )

    print("Calibration successful!")
    print("Camera matrix (intrinsics):\n", camera_matrix) # camera_matrix = K from discussion
    print("Distortion coefficients:\n", dist_coeffs)

else:
    print("No valid markers detected in any image. Calibration unsuccessful.")

Using 227 detected tags for calibration
Calibration successful!
Camera matrix (intrinsics):
 [[4.89944684e+03 0.00000000e+00 1.51103357e+03]
 [0.00000000e+00 4.56223849e+03 2.00263531e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coefficients:
 [[ 1.00048005 -7.65139338  0.11521363 -0.08588187 23.52314764]]


In [ ]:
# # test if programs work on spec's sample data
# # calibration pipeline

# # starter code from spec
# # import cv2
# # import numpy as np

# # Create ArUco dictionary and detector parameters (4x4 tags)
# aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
# aruco_params = cv2.aruco.DetectorParameters()

# # tag size
# tag_size = 0.06 # meters, 0.02 = 2 cm
# # 3d coordinates of aruco tag in world frame
# obj_pts_single = np.array([
#     [0, 0, 0],           # Top-left
#     [tag_size, 0, 0],    # Top-right
#     [tag_size, tag_size, 0],  # Bottom-right
#     [0, tag_size, 0]     # Bottom-left
# ], dtype = np.float32)

# obj_pts = [] # all 3D pts in world frame
# img_pts = [] # all 2D pts in image frame

# images = glob.glob('calibration-images/*.JPG') # path to images
# # images = glob.glob('desk-calibration-images/*.jpg') # path to images, path file ending is case sensitive (e.g. jpg vs JPG are different)

# for path in images:
#     # load image
#     im = cv2.imread(path)
#     gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

#     # Detect ArUco markers in an image
#     # Returns: corners (list of numpy arrays), ids (numpy array)
#     corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params) # modify to take in grayscale image

#     # Check if any markers were detected
#     if ids is None or len(ids) == 0:
#         print(f"No ArUco detected in {path}, skipping.")
#         continue
#     # Process the detected corners
#     # corners: list of length N (number of detected tags)
#     #   - each element is a numpy array of shape (1, 4, 2) containing the 4 corner coordinates (x, y)
#     # ids: numpy array of shape (N, 1) containing the tag IDs for each detected marker
#     # Example: if 3 tags detected, corners will be a list of 3 arrays, ids will be shape (3, 1)
#         # loop through all detected markers
#     for c in corners:
#         img_pts.append(c.reshape(4, 2)) # reshape from (1, 4, 2) to shape: (4,2)
#         obj_pts.append(obj_pts_single) # append the same 3D points for each detected marker

# # 5. calibrate camera
# if len(obj_pts) >= 10:
#     print(f"Using {len(obj_pts)} detected tags for calibration")
#     img_shape = gray.shape[::-1] # (width, height)
    
#     ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
#         obj_pts,
#         img_pts,
#         img_shape,
#         None, # initial camera matrix (estimated by OpenCV)
#         None # initial distortion coefficients
#     )

#     print("Calibration successful!")
#     print("Camera matrix (intrinsics):\n", camera_matrix) # camera_matrix = K from discussion
#     print("Distortion coefficients:\n", dist_coeffs)

# else:
#     print("No valid markers detected in any image. Calibration unsuccessful.")

Using 204 detected tags for calibration
Calibration successful!
Camera matrix (intrinsics):
 [[3.58646600e+03 0.00000000e+00 2.94782890e+03]
 [0.00000000e+00 3.08487288e+03 2.20427722e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coefficients:
 [[ 0.19103245 -1.20355492  0.03966894  0.00369337  1.59863158]]


### Part 0.3: Estimating Camera Pose

In [9]:
# visualize
server = viser.ViserServer(share=True)

obj_images = glob.glob("octo-images/*.jpg")
obj_images = sorted(obj_images)

# for part 0.4
undistorted_images = []
c2w_matrices = []

# for each image, detect the single aruco tag and use cv2.solvePnP() to estimate the camera pose
for i, path in enumerate(obj_images): # rn images = calibration images
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params)
    if ids is None or len(ids) == 0:
        print(f"No ArUco detected in {path}, skipping.") # skip as before in part 0.1
        continue

    # use first marker
    img_pts_single = corners[0].reshape(4, 2).astype(np.float32) # reshape corners

    # outputs = solvePnP(inputs)
    success, rvec, tvec = cv2.solvePnP(obj_pts_single, img_pts_single, camera_matrix, dist_coeffs)
    if not success:
        print(f"PnP failed for {path}, skipping.") # check what's happening
        continue

    R, _ = cv2.Rodrigues(rvec) # convert to 3x3 rotation matrix
    t = tvec.reshape(3, 1)
    R_wc = R.T
    t_wc = -R.T @ t
    c2w = np.hstack([R_wc, t_wc]) # invert world-to-camera to get camera-to-world

    # undistort the image
    undistorted = cv2.undistort(img, camera_matrix, dist_coeffs)
    # convert BGR to RGB for NeRF
    undistorted = undistorted[:, :, ::-1]
    # append/save
    undistorted_images.append(undistorted)
    c2w_full = np.eye(4)
    c2w_full[:3, :4] = c2w
    c2w_matrices.append(c2w_full)

    # convert BGR to RGB for viser
    img_rgb = img[:, :, ::-1]

    server.scene.add_camera_frustum(
        f"/cameras/{i}",
        fov=2 * np.arctan2(gray.shape[0] / 2, camera_matrix[0, 0]),
        aspect=gray.shape[1] / gray.shape[0],
        scale=0.02,
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=img_rgb
    )


╭────── viser (listening *:8082) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8082   │
│   Websocket │ ws://localhost:8082     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://warp-sample.share.viser.studio

(viser) Connection opened (2, 1 total), 138 persistent messages

(viser) Connection closed (2, 0 total)

(viser) Connection opened (0, 1 total), 166 persistent messages

(viser) Connection opened (1, 2 total), 166 persistent messages

In [ ]:
# # visualize
# server = viser.ViserServer(share=True)

# obj_images = glob.glob("lafufu-images/*.JPG")
# obj_images = sorted(obj_images)

# # for each image, detect the single aruco tag and use cv2.solvePnP() to estimate the camera pose
# for i, path in enumerate(obj_images): # rn images = calibration images
#     img = cv2.imread(path)
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

#     corners, ids, _ = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=aruco_params)
#     if ids is None or len(ids) == 0:
#         print(f"No ArUco detected in {path}, skipping.") # skip as before in part 0.1
#         continue

#     # use first marker
#     img_pts_single = corners[0].reshape(4, 2).astype(np.float32) # reshape corners

#     # outputs = solvePnP(inputs)
#     success, rvec, tvec = cv2.solvePnP(obj_pts_single, img_pts_single, camera_matrix, dist_coeffs)
#     if not success:
#         print(f"PnP failed for {path}, skipping.") # check what's happening
#         continue

#     R, _ = cv2.Rodrigues(rvec) # convert to 3x3 rotation matrix
#     t = tvec.reshape(3, 1)
#     R_wc = R.T
#     t_wc = -R.T @ t
#     c2w = np.hstack([R_wc, t_wc]) # invert world-to-camera to get camera-to-world

#     # convert BGR to RGB for viser
#     img_rgb = img[:, :, ::-1]

#     server.scene.add_camera_frustum(
#         f"/cameras/{i}",
#         fov=2 * np.arctan2(gray.shape[0] / 2, camera_matrix[0, 0]),
#         aspect=gray.shape[1] / gray.shape[0],
#         scale=0.02,
#         wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
#         position=c2w[:3, 3],
#         image=img_rgb
#     )


╭────── viser (listening *:8081) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8081   │
│   Websocket │ ws://localhost:8081     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://unreliable-electric.share.viser.studio

(viser) Connection closed (2, 1 total)

(viser) Connection closed (0, 0 total)

(viser) Connection opened (0, 1 total), 138 persistent messages

(viser) Connection opened (1, 2 total), 138 persistent messages

### Part 0.4: Undistorting images and creating a dataset

In [ ]:
# after for loop to solve PnP + viser
undistorted_images = np.array(undistorted_images) # (N, H, W, 3)
c2w_matrices = np.array(c2w_matrices) # (N, 4, 4)

# train/test split
# train/val/test = 80/10/10
N = len(undistorted_images)
idx_train = int(0.8 * N)
idx_val = int(0.9 * N)

images_train = undistorted_images[:idx_train]
c2ws_train   = c2w_matrices[:idx_train]

images_val   = undistorted_images[idx_train:idx_val]
c2ws_val     = c2w_matrices[idx_train:idx_val]

c2ws_test    = c2w_matrices[idx_val:]

# use average of fx and fy to extract focal length
focal = float((camera_matrix[0, 0] + camera_matrix[1, 1]) / 2)

# from spec to save dataset
# Package your data (keep images in 0-255 range, they'll be normalized when loaded)
np.savez(
    'my_data.npz',
    images_train=images_train,    # (N_train, H, W, 3)
    c2ws_train=c2ws_train,        # (N_train, 4, 4)
    images_val=images_val,        # (N_val, H, W, 3)
    c2ws_val=c2ws_val,            # (N_val, 4, 4)
    c2ws_test=c2ws_test,          # (N_test, 4, 4)
    focal=focal                   # float
)

print("Saved dataset to my_data.npz")

Saved dataset to octo_data.npz


## Part 1: Fit a Neural Field to a 2D Image